# Laboratorio 6
Alina Carías, Daniel Machic, Ariela Mishaan

**Github:** https://github.com/ArielaMishaanCohen/LAB6.git

## Task 1

### Ejercicio 1

#### Por qué una red tipo VGG de 150 capas fracasaría

El argumento de "más prfundo siempre es mejor" es intuitivo, pero incorrecto. Una red secuencial VGGde 150 capas fallaría por dos razones: 

**1. Desvanecimiento del gradiente**
Durante el entrenamiento, los pesos se actualizan usando backpropagation: 

$$\Delta w = -\eta \frac{\partial{J}}{\partial{w}}$$

donde J es la función de pérdida, w los pesos y $\eta$ el learning rate. 

El gradiente, mientras más capas lleve la red neuronal, es un producto de más derivadas. Si se usan activaciones como ReLU o funciones con derivadas menores que 1, el resultado de la multiplicación será cada vez más cercano a cero. Por eso se le llama "vanishing gradient". Cuando la red es muy profunda, los pesos terminan siendo 0 en las últimas capas, lo que provoca que la red ya no siga aprendiendo. El entrenamiento se vuelve muy lento y el modelo se estanca. 

**2. Fenómeno de degradación**

Teóricamente, al agregar más capas, el error de entrenamiento no debería empeorar, porque las nuevas capas podrían aprender la identidad: 

$$H(x) = x$$

Sin embargo, en redes profundas secuenciales ocurre lo contrario. Al aumentar la profundidad, el error de entrenamiento aumenta. El modelo profundo rinde peor que uno menos profundo, no por sobreajuste, sino por problema de optimización. Esto pasa porque la red debe aprender directamente una función compleja: $H(x)$, y al aumentar capas, el espacio de optimización se vuelve más difícil de recorrer. 

**Conclusión:** entrenar una VGG-150 sería inestable, muy costoso, propenso a colapsar y no funcionaría

#### Cómo ResNet rescata el proyecto

ResNet introduce una modificación a la estructura: 

$$y = F(x) + x$$

En lugar de aprender directamente $H(x)$, la red aprende el residuo: 

$$F(x) = H(x) - x$$

Ahora el gradiente en backpropagation no tiende a 0:

$$\frac{\partial y}{\partial x} = \frac{\partial F(x)}{\partial x} + 1$$

Gracias al +1, el gradiente no desaparece, als capas siguen aprendiendo y la red puede ser profunda. 

Además, el problema de optimización se vuelve más fácil. Si la función óptima es cercana a la identidad, ahora la red solo debe aprender $F(x) \sim 0$, qeu es más sencillo que forzar múltiples capas a aprender identidad implícitamente. 

### Ejercicio 2

#### Por qué inception es ideal para enfermedades visualmente heterogéneas

El problema que se quiere resolver tiene dos aspectos importantes: 
- **Antracnosis**: puntos negros muy pequeños
- **Moho polvoriento**: áreas muy grandes

Esto muestra que las características que se quieren identificar ocurren en múltiples escalas espaciales. 

**Limitación de una CNN secuencial tradicional**
En una CNN normal, cada capa aplica solo un tamaño de filtro (3x3). O sea, cada nivel de la red captura información con solo un campo receptivo. Dado que el problema tiene dos dimensiones importantes, se necesitan: 
- campos receptivos pequeños para detectar antracnosis
- campos receptivos grandes para el moho polvoriento

**Cómo lo resuelve Inception**
Inception procesa la entrada en paralelo con distintos tamaños de filtros (1x1, 3x3, 5x5) y luego concatena los resultados. 

Entonces, en una capa:
- El filtro 3x3 captura texturas locales
- El 5x5 captura estructuras más amplias
- El 1x1 permite interacción entre canales. 

En nuestro problema, el filtro 3x3 identifica la Antracnosis, el 5x5 el moho y la concatenación permite que el clasificador final deciad qué escala es más relevante. 

#### Complejidad computacional

El problema es que usar múltiples filtros grandes aumenta dramáticamente el costo. El costo de una convolución estándar es: 

$$Costo = k^2 \cdot C_{in} \cdot C_{out} \cdot H \cdot W$$

Un filtro 5x5 es:
$$25 \cdot C_{in} \cdot C_{out}$$

Comparado con un 3x3:
$$9 \cdot C_{in} \cdot C_{out}$$

Si usamos varios en paralelo, el número de parámetros se hace demasiado grande. Esto implica más memoria GPU, más tiempo de entrenamiento, más costo económico y más consumo de energía. 

**Convoluciones 1x1**
Inception, antes de aplicar los filtros 3x3 y 5x5, que son costosos, reduce la dimensionalidad usando una convolución 1x1. 

Una convolución 1x1 cuesta 

$$1 \cdot C_{in} \cdot C_{reducido}$$

y transforma 

$$C_{in} \to C_{reducido}$$

Entonces, el costo del 5x5 pasa de 

$$25 \cdot C_{in} \cdot C_{out}$$

a 

$$1 \cdot C_{in} \cdot C_{reducido} + 25 \cdot C_{reducido} \cdot C_{out}$$

Si $C_{reducido} << C_{in}$, entonces la reducción de parámetros es muy grande. 

### Ejercicio 3

#### Cómo funciona la Depthwise Separable Convolution

**Convolución estándar**  
Una convolución tradicional hace simultáneamente filtrado espacial y mezcla de canales. Su costo es: 

$$k^2 \cdot C_{in} \cdot C_{out} \cdot H \cdot W$$

Si se usa un filtro 3x3: 

$$9 \cdot C_{in} \cdot C_{out}$$

Esto crece rápidamente cuando se aumentan los canales. 

**Depthwise Separable Convolution**  
MobileNet logra su eficiencia dividiendo la convolución estándar en dos pasos.

- **Depthwise convolution (filtrado espacial)**: aplica un filtro $k \times k$ por cada canal. No mezcla canales. Su costo es: 
  
  $$k^2 \cdot C_{in} \cdot H \cdot W$$

- **Pointwise Convolution (1x1)**: mezcla los canales. Su costo es: 
  
  $$C_{in} \cdot C_{out} \cdot H \cdot W$$

El costo total de la Depthwise Separable Convolution es: 

$$k^2 \cdot C_{in} + C_{in} \cdot C_{out}$$

Comparado con la estándar: 

$$k^2 \cdot C_{in} \cdot C_{out}$$

**Reducción**  
Para un filtro 3x3:

$$\frac{9C_{in} + C_{in} C_{out}}{9C_{in} C_{out}}$$

Cuando $C_{out}$ es grande, la reducción puede ser cercana al 90% menos de cómputo, lo que implica menos memoria, menos operaciones, menor consumo y más rapidez, que es lo que busca la startup. 

#### Qué estamos sacrificando matemáticamente

En una convolución estándar, cada filtro aprende simultáneamente patrón espacial + combinación inter-canal. El kernel completo tiene libertad para modelar correlaciones espaciales y entre canales al mismo tiempo.  

En la versión separada primero se aprende patrón espacial por canal y luego se mezclan canales linealmente. Esto causa una restricción en la estructura: la operación ya no es completamente libre en el espacio $k \times k \times C_{in}$. En términos matemáticos, se está factorizando una operación de alto rango en dos operaicones de menor rango. La consecuencia es menor capacidad expresiva, menor interacción compleja entre canales en una sola etapa y posible ligera pérdida de accuracy. 

#### Por qué se acepta el costo

En el contexto del proyecto de la startup (teléfonos de gama baja con poca RAM, sin GPU dedicada ni conexión a la nube), si se elige un modelo pesado, la app se vuelve lenta. El teléfono podría llegar a cerrar la app por falta de memoria. Además la batería se drena y no es cómodo para el usuario. 